In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")
include("mglm_utils.jl")

default_plot_setting()

In [ ]:
df, df_part = read_comix_uk_contact_raw();

println("# contacts: ", nrow(df))
println("# participants (rows): ", nrow(df_part))

In [ ]:
df_chunk = create_week_df(df_part)
df_part = add_date_chunks(@subset(df_part,
    Date(2021, 7, 1) .<= :date .<= Date(2021, 12, 31)))

df = innerjoin(df, @select(df_part, :part_id_d, :date,
    :chunk_number, :chunk_start, :chunk_end, :mid_date),
    on = [:part_id_d, :date])

println("# weeks: ", nrow(df_chunk))
first(df_chunk, 5)

In [ ]:
is_na(v) = ismissing(v) || (v isa AbstractString && v == "NA")

n_total       = nrow(df)
n_miss_phys   = count(is_na, df[:, :phys_contact])
n_miss_dur    = count(is_na, df[:, :duration_multi])
n_home        = count(==("true"),  df[:, :cnt_home])
n_nonhome     = count(==("false"), df[:, :cnt_home])

println("contacts total           : ", n_total)
println("  missing phys_contact   : ", n_miss_phys)
println("  missing duration_multi : ", n_miss_dur)
println("  cnt_home == true       : ", n_home)
println("  cnt_home == false      : ", n_nonhome)

# Visualisation of proportions

In [ ]:
# Aggregated empirical proportions vs degree, three duration variants + phys.
# Helpers live in `vis_utils.jl` (included by `cell-setup`).
#   1. NA imputed as <5 min  (matches `prepare_dm_inputs` default).
#   2. NA kept as its own 6th category.
#   3. NA excluded from proportion (numerator AND denominator), but degree
#      (x-axis) still counts NA-duration contacts.
p_dur_home        = _plot_props_panel(df, "home",     :duration_multi, 5; title_suffix = " (NA→<5min)")
p_dur_non         = _plot_props_panel(df, "non-home", :duration_multi, 5; title_suffix = " (NA→<5min)")
p_dur_na_home     = _plot_props_panel_dur_na(df, "home")
p_dur_na_non      = _plot_props_panel_dur_na(df, "non-home")
p_dur_dropna_home = _plot_props_panel_dur_dropna(df, "home")
p_dur_dropna_non  = _plot_props_panel_dur_dropna(df, "non-home")
p_phys_home       = _plot_props_panel(df, "home",     :phys_contact,   2)
p_phys_non        = _plot_props_panel(df, "non-home", :phys_contact,   2)

fig_props_agg = plot(p_dur_home, p_dur_non,
     p_dur_na_home, p_dur_na_non,
     p_dur_dropna_home, p_dur_dropna_non,
     p_phys_home, p_phys_non;
    layout = (4, 2), size = (1100, 1400))

out_dir_fig = "../res/2j_proportion_duration_physical"
isdir(out_dir_fig) || mkpath(out_dir_fig)
savefig(fig_props_agg, joinpath(out_dir_fig, "props_aggregated.png"))

fig_props_agg

In [ ]:
# Raw per-cell proportions vs degree (no degree-aggregation).
# Helpers live in `vis_utils.jl` (included by `cell-setup`).
# Marker area ∝ log10(# cells stacked at that exact (degree, proportion) point).

# Duration: 5 categories × 2 settings (NA imputed as <5min).
plts_dur_raw = Plots.Plot[]
for k in 1:5
    push!(plts_dur_raw, _plot_raw_panel(df, "home",     :duration_multi, 5, k))
    push!(plts_dur_raw, _plot_raw_panel(df, "non-home", :duration_multi, 5, k))
end
fig_dur_raw = plot(plts_dur_raw...; layout = (5, 2), size = (1100, 1500),
    plot_title = "Raw per-cell duration proportions vs degree (NA→<5min)")

# Duration: 6 categories × 2 settings (NA as its own category).
plts_dur_raw_na = Plots.Plot[]
for k in 1:6
    push!(plts_dur_raw_na, _plot_raw_panel_dur_na(df, "home",     k))
    push!(plts_dur_raw_na, _plot_raw_panel_dur_na(df, "non-home", k))
end
fig_dur_raw_na = plot(plts_dur_raw_na...; layout = (6, 2), size = (1100, 1800),
    plot_title = "Raw per-cell duration proportions vs degree (NA as category)")

# Physical: 2 categories × 2 settings.
plts_phys_raw = Plots.Plot[]
for k in 1:2
    push!(plts_phys_raw, _plot_raw_panel(df, "home",     :phys_contact, 2, k))
    push!(plts_phys_raw, _plot_raw_panel(df, "non-home", :phys_contact, 2, k))
end
fig_phys_raw = plot(plts_phys_raw...; layout = (2, 2), size = (1100, 600),
    plot_title = "Raw per-cell physical-contact proportions vs degree")

out_dir_fig = "../res/2j_proportion_duration_physical"
isdir(out_dir_fig) || mkpath(out_dir_fig)
savefig(fig_dur_raw,    joinpath(out_dir_fig, "props_raw_duration.png"))
savefig(fig_dur_raw_na, joinpath(out_dir_fig, "props_raw_duration_na.png"))
savefig(fig_phys_raw,   joinpath(out_dir_fig, "props_raw_physical.png"))

display(fig_dur_raw)
display(fig_dur_raw_na)
display(fig_phys_raw)

# Dirichlet-multinomial regression: contact duration & physical contact proportions

Analysis for 2021-07 to 2021-12, separately for home vs non-home contacts.

- Outcome 1 — `duration_multi` (K=5): <5min, 5–15min, 15min–1hr, 1–4hr, 4+hr.
- Outcome 2 — `phys_contact` (K=2): physical, non-physical.
- Predictor: `log(degree)` per (participant, diary day, setting).
- Fitter: R `MGLM::MGLMreg(..., dist = "DM")` invoked through the
  `fit_mglm_dm` wrapper in `mglm_utils.jl` (RCall).
- Model comparison: AIC/BIC of the unrestricted fit vs an intercept-only null.

In [ ]:
# Primary policy: impute missing duration as <5 min, drop missing phys_contact.
panels = Dict{Symbol, NamedTuple}()
for (panel, setting, outcome, K) in [
        (:dur_home,     "home",     :duration_multi, 5),
        (:dur_nonhome,  "non-home", :duration_multi, 5),
        (:phys_home,    "home",     :phys_contact,   2),
        (:phys_nonhome, "non-home", :phys_contact,   2),
    ]
    panels[panel] = prepare_dm_inputs(df; setting = setting, outcome = outcome, K = K)
    inp = panels[panel]
    println(panel, " -> N=", size(inp.X, 1), ", K=", K,
        ", n range=", extrema(inp.n))
end

In [ ]:
K_for_panel = Dict(:dur_home => 5, :dur_nonhome => 5,
                   :phys_home => 2, :phys_nonhome => 2)

In [ ]:
# Unrestricted MGLM Dirichlet-multinomial fit per panel.
mglm_full = Dict{Symbol, NamedTuple}()
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    println("Fitting full MGLM-DM for ", panel, " (N=", size(inp.X, 1), ", K=", K, ")")
    mglm_full[panel] = fit_mglm_dm(inp.X, inp.Y)
end

In [ ]:
# Intercept-only DM fit per panel (AIC/BIC null for the log(degree) term).
mglm_null = Dict{Symbol, NamedTuple}()
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    println("Fitting null MGLM-DM for ", panel)
    mglm_null[panel] = fit_mglm_dm(inp.X, inp.Y; intercept_only = true)
end

In [ ]:
# Per-panel diagnostic summary (coefficients, SEs, Wald test, AIC/BIC).
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    println("=== ", panel, " — full ===")
    mglm_dm_show(mglm_full[panel])
    println("=== ", panel, " — null ===")
    mglm_dm_show(mglm_null[panel])
end

In [ ]:
# Model comparison: unrestricted vs intercept-only DM (positive Δ favours full).
aic_rows = NamedTuple[]
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    cmp = mglm_compare(mglm_full[panel], mglm_null[panel])
    push!(aic_rows, (panel = panel,
        AIC_full = mglm_full[panel].AIC, AIC_null = mglm_null[panel].AIC,
        BIC_full = mglm_full[panel].BIC, BIC_null = mglm_null[panel].BIC,
        ΔAIC = cmp.ΔAIC, ΔBIC = cmp.ΔBIC, prefer = cmp.prefer))
end
df_aic = DataFrame(aic_rows)
df_aic

In [ ]:
# Empirical proportions by degree — used by the predicted-curve plot below.
function empirical_proportions_by_n(Y::Matrix{Int}, n::Vector{Int}, K::Int)
    df_e = DataFrame(n = n)
    for k in 1:K; df_e[!, Symbol("y$k")] = Y[:, k]; end
    g = combine(groupby(df_e, :n)) do sub
        total = sum(sub[:, :n])
        (; (Symbol("p$k") => sum(sub[:, Symbol("y$k")]) / total for k in 1:K)...,
           ncell = nrow(sub))
    end
    sort!(g, :n)
    return g
end

In [ ]:
# Empirical aggregated mean proportions vs log(degree), with the predicted
# DM marginal overlaid: a 95% band on Y_k/n for an individual at degree n,
# computed analytically from the BetaBinomial marginal of the fitted
# Dirichlet-multinomial — Var(Y_k/n) = μ_k (1−μ_k) (n + α₀) / (n (1 + α₀)),
# then μ_k ± 1.96·σ clamped to [0, 1]. Uses the AIC-preferred model per panel.
n_grid = collect(1:50)
category_names = Dict(
    :duration_multi => ["<5min", "5–15min", "15min–1hr", "1–4hr", "4+hr"],
    :phys_contact   => ["physical", "non-physical"])

panel_outcome = Dict(:dur_home => :duration_multi, :dur_nonhome => :duration_multi,
                     :phys_home => :phys_contact,  :phys_nonhome => :phys_contact)

pp_plots = Dict{Symbol, Plots.Plot}()
for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    inp = panels[panel]; K = K_for_panel[panel]
    chosen = df_aic[df_aic.panel .== panel, :prefer][1]
    fit = chosen == "null" ? mglm_null[panel] : mglm_full[panel]

    pred = mglm_dm_predict(fit, n_grid)
    emp  = empirical_proportions_by_n(inp.Y, inp.n, K)
    names = category_names[panel_outcome[panel]]
    log_n_grid = log.(n_grid)

    plts = Plots.Plot[]
    for k in 1:K
        p = plot(log_n_grid, pred.μ[:, k];
            ribbon = (pred.μ[:, k] .- pred.lower[:, k],
                      pred.upper[:, k] .- pred.μ[:, k]),
            xlabel = "log(degree)", ylabel = "proportion",
            title  = string(names[k]),
            label  = "DM mean ± 95% pred band",
            lw = 2, fillalpha = 0.18)
        scatter!(p, log.(emp.n), emp[!, Symbol("p$k")];
            ms = log.(emp.ncell),
            label = "empirical mean (per degree)",
            markerstrokewidth = 0)
        push!(plts, p)
    end
    pp_plots[panel] = plot(plts...; layout = (1, K), size = (340 * K, 300),
        plot_title = string(panel, " — MGLM (", chosen, ")"))
    display(pp_plots[panel])
end

out_dir_fig = "../res/2j_proportion_duration_physical"
isdir(out_dir_fig) || mkpath(out_dir_fig)
for (panel, p) in pp_plots
    savefig(p, joinpath(out_dir_fig, "pp_$(panel).png"))
end

In [ ]:
# Sensitivity: drop ALL rows where either outcome is missing.
panels_sens = Dict{Symbol, NamedTuple}()
mglm_full_sens = Dict{Symbol, NamedTuple}()
mglm_null_sens = Dict{Symbol, NamedTuple}()
aic_rows_sens = NamedTuple[]
for (panel, setting, outcome, K) in [
        (:dur_home,     "home",     :duration_multi, 5),
        (:dur_nonhome,  "non-home", :duration_multi, 5),
        (:phys_home,    "home",     :phys_contact,   2),
        (:phys_nonhome, "non-home", :phys_contact,   2),
    ]
    inp = prepare_dm_inputs(df; setting = setting, outcome = outcome, K = K,
        drop_all_missing = true)
    panels_sens[panel] = inp
    mglm_full_sens[panel] = fit_mglm_dm(inp.X, inp.Y)
    mglm_null_sens[panel] = fit_mglm_dm(inp.X, inp.Y; intercept_only = true)
    cmp = mglm_compare(mglm_full_sens[panel], mglm_null_sens[panel])
    push!(aic_rows_sens, (panel = panel, N = size(inp.X, 1),
        AIC_full = mglm_full_sens[panel].AIC, AIC_null = mglm_null_sens[panel].AIC,
        BIC_full = mglm_full_sens[panel].BIC, BIC_null = mglm_null_sens[panel].BIC,
        ΔAIC = cmp.ΔAIC, ΔBIC = cmp.ΔBIC, prefer = cmp.prefer))
end
df_aic_sens = DataFrame(aic_rows_sens)
df_aic_sens

In [ ]:
# Save MGLM fits (as .rds, readable from R) and AIC/BIC summaries.
out_dir = "../res/2j_proportion_duration_physical"
isdir(out_dir) || mkpath(out_dir)

for panel in [:dur_home, :dur_nonhome, :phys_home, :phys_nonhome]
    save_mglm_fit(mglm_full[panel],      joinpath(out_dir, "mglm_$(panel)_full.rds"))
    save_mglm_fit(mglm_null[panel],      joinpath(out_dir, "mglm_$(panel)_null.rds"))
    save_mglm_fit(mglm_full_sens[panel], joinpath(out_dir, "mglm_$(panel)_full_sens.rds"))
    save_mglm_fit(mglm_null_sens[panel], joinpath(out_dir, "mglm_$(panel)_null_sens.rds"))
end

CSV.write(joinpath(out_dir, "aic_primary.csv"),     df_aic)
CSV.write(joinpath(out_dir, "aic_sensitivity.csv"), df_aic_sens)
println("Saved to ", out_dir)